# Apache Hive

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/hive/hive.ipynb)

Apache Hive turns SQL into distributed jobs: you declare schemas and partitions over files in HDFS or object storage, and the engine plans and executes the query across nodes. The questions below cover the storage model, partitioning, file formats, joins and the query lifecycle, with HQL you can adapt to any warehouse.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

The runnable cells use DuckDB as an in-process stand-in for Hive: the same layout ideas (partition directories, Parquet, bucketing, complex types) apply, without a cluster.

## Architecture and the metastore

### Components and the query lifecycle

The client submits SQL. The driver parses it, the compiler builds a logical plan, the metastore supplies schema and statistics, the optimizer turns the plan into a physical plan, and the execution engine runs it. On Hadoop that engine is MapReduce, Tez, or Spark. Results are fetched back through the driver or HiveServer2.

```text
client (CLI, beeline, JDBC)
   -> driver -> compiler -> optimizer -> execution engine
                     |                        |
                 metastore                 HDFS / S3
```
Common mistake: blaming Hive for a slow query when the time is spent in one reducer. The plan view in `EXPLAIN` shows which stage runs how many tasks; that is where to look first.

The pipeline is parse, analyze, optimize, plan, execute. The parser turns SQL into an abstract syntax tree. The semantic analyzer resolves names against the metastore and checks types. The logical optimizer applies rules such as predicate pushdown and partition pruning. The physical planner picks join and aggregation strategies, and the execution engine runs the resulting tasks.

### The metastore

The metastore is a relational database plus a Thrift service. The database (Derby, MySQL, or PostgreSQL) holds schemas such as `DBS`, `TBLS`, `COLUMNS_V2`, `PARTITIONS`, `SDS`, and `TAB_COL_STATS`. What is stored is metadata: table and column names, types, SerDe class, input and output formats, the storage location, the partition list, statistics, and grants. The data stays in HDFS or object storage. That split is exactly what schema-on-read means: the table is a definition over files.

```bash
# Run the metastore as its own service so many clients share one catalog
hive --service metastore
# Clients then point at it instead of a local embedded database
beeline -u jdbc:hive2://hive-ms:10000/default
```
Common mistake: leaving the default embedded Derby metastore in place for a shared cluster. Derby allows one writer at a time, so a second client fails. Production uses an external RDBMS behind a remote metastore service.

```xml
<!-- hive-site.xml: remote metastore backed by MySQL -->
<configuration>
  <property>
    <name>javax.jdo.option.ConnectionURL</name>
    <value>jdbc:mysql://mysql1:3306/hive?createDatabaseIfNotExist=true</value>
  </property>
  <property>
    <name>javax.jdo.option.ConnectionDriverName</name>
    <value>com.mysql.cj.jdbc.Driver</value>
  </property>
  <property>
    <name>hive.metastore.uris</name>
    <value>thrift://hive-ms:9083</value>
  </property>
  <property>
    <name>hive.metastore.warehouse.dir</name>
    <value>/user/hive/warehouse</value>
  </property>
</configuration>
```

The metastore has three configurations. Embedded uses a Derby database inside the Hive driver, ideal only for a single-user sandbox. Local runs an external database but keeps the metastore service inside the driver. Remote runs the metastore as its own Thrift service, which is what a shared cluster needs. Separately, Hive can be driven through the legacy CLI, through HiveServer2 with beeline or JDBC, or embedded in an application.

### Data model

The model has four levels. A database is a namespace. A table maps to a directory. A partition maps to a subdirectory and is registered in the metastore. A bucket maps to a file inside the partition. Columns, types, and the SerDe describe how to read the bytes at the bottom.

```sql
SHOW DATABASES;
SHOW TABLES IN retail;
SHOW PARTITIONS retail.orders;
DESCRIBE FORMATTED retail.orders;
```
Common mistake: thinking of partitions as indexes. They are directories. If data is written straight into HDFS without `MSCK REPAIR TABLE` or an `ALTER TABLE ... ADD PARTITION`, the metastore never learns about it and Hive will not read it.

### Schema on read, managed and external tables

Schema on read means the table definition is applied when data is queried, not when it is written. Hive stores no parsed rows; it stores column names, types, and a SerDe in the metastore, and the reader interprets the bytes at scan time. This is why a malformed file fails or returns nulls only when queried, and why several tables can describe the same files differently.

```sql
-- The same files, two schemas
CREATE EXTERNAL TABLE raw_v1 (a STRING, b STRING)
LOCATION '/data/events/';

CREATE EXTERNAL TABLE raw_v2 (a STRING, b STRING, c STRING)
LOCATION '/data/events/';
```
Common mistake: treating schema on read as free. There is no write-time validation, so bad rows are discovered by the analyst rather than the loader. Validate against a contract at ingest, or use an ACID table where schema and statistics are maintained.

A managed table owns its data. The files live under the warehouse directory for that database, and `DROP TABLE` deletes them. An external table only points at a location: `DROP TABLE` removes the metadata and leaves the files. The distinction is about lifecycle ownership, not about format or performance.

```sql
CREATE TABLE managed_orders (order_id BIGINT, amount DECIMAL(12,2))
STORED AS ORC;

CREATE EXTERNAL TABLE external_orders (order_id BIGINT, amount DECIMAL(12,2))
STORED AS ORC
LOCATION 's3://lake/orders/';

DROP TABLE managed_orders;    -- files deleted
DROP TABLE external_orders;   -- files survive
```
Common mistake: using managed tables for data that other engines also read. Drop or truncate then destroys shared files. Land raw data as external and build curated tables as managed, or mark every shared table external.

The main split is managed versus external, which decides who owns the files. Then there are views, materialized views where supported, temporary tables that live only for a session, and ACID transactional tables that support `UPDATE`, `DELETE`, and `MERGE`. Virtual tables such as `DUAL`-style helpers exist for ad hoc queries, and archive tables matter only in specific setups.

```sql
CREATE TABLE managed_t  (id BIGINT) STORED AS ORC;
CREATE EXTERNAL TABLE ext_t (id BIGINT) STORED AS ORC LOCATION 's3://lake/ext/';
CREATE TEMPORARY TABLE tmp_t AS SELECT * FROM managed_t WHERE id > 10;
CREATE VIEW v_t AS SELECT id FROM managed_t;
CREATE TABLE acid_t (id BIGINT, v STRING)
CLUSTERED BY (id) INTO 8 BUCKETS STORED AS ORC
TBLPROPERTIES ('transactional' = 'true');
```
Common mistake: using a temporary table across statements in different sessions. It exists only for the connection that created it, so a later beeline call cannot see it. Use a real staging table when other steps must read it.

The cells below build a small warehouse on local disk. The first shows schema on read and the managed versus external lifecycle.

In [ ]:
import tempfile
from pathlib import Path

import duckdb

con = duckdb.connect()
WAREHOUSE = Path(tempfile.mkdtemp(prefix="hive_warehouse_"))

con.execute("""
    CREATE TABLE staging_orders AS
    SELECT i AS order_id,
           i % 50 AS customer_id,
           round(10 + (i % 97) * 1.5, 2) AS amount,
           DATE '2026-10-01' + CAST(i % 5 AS INTEGER) AS order_date
    FROM range(1000) t(i)
""")

# INSERT OVERWRITE ... PARTITION (order_date): one directory per value, Hive-style key=value names.
orders_dir = WAREHOUSE / "orders"
con.execute(f"COPY staging_orders TO '{orders_dir}' (FORMAT parquet, PARTITION_BY (order_date))")
partitions = sorted(p.name for p in orders_dir.iterdir())
print(partitions)
assert partitions[0] == "order_date=2026-10-01" and len(partitions) == 5

In [ ]:
raw_dir = WAREHOUSE / "raw_events"
raw_dir.mkdir()
(raw_dir / "part-00000.csv").write_text("1,click,2026-10-04\n2,view,2026-10-04\n3,click,not-a-date\n")

# EXTERNAL TABLE: a definition over files. Two schemas can describe the same bytes.
con.execute(f"""
    CREATE VIEW raw_text AS
    SELECT * FROM read_csv('{raw_dir}/*.csv', header = false,
                           columns = {{'event_id': 'BIGINT', 'event_type': 'VARCHAR', 'event_day': 'VARCHAR'}})
""")
con.execute("CREATE VIEW raw_typed AS SELECT event_id, event_type, TRY_CAST(event_day AS DATE) AS event_day FROM raw_text")
bad_rows = con.execute("SELECT event_id FROM raw_typed WHERE event_day IS NULL").fetchall()
assert bad_rows == [(3,)]  # nothing validated the file at load time; the bad row surfaces at query time

# MANAGED TABLE: the engine owns a copy. Dropping the external definitions leaves the files.
con.execute("CREATE TABLE managed_events AS SELECT * FROM raw_typed WHERE event_day IS NOT NULL")
con.execute("DROP VIEW raw_typed")
con.execute("DROP VIEW raw_text")
assert (raw_dir / "part-00000.csv").exists()
assert con.execute("SELECT count(*) FROM managed_events").fetchone()[0] == 2
print("bad rows found at read time:", bad_rows, "| files survive DROP of the external definition")

## Storage layout

### Partitioning versus bucketing

A partition is a directory, and the metastore keeps the list of them. The optimizer can therefore drop whole partitions before a single file is opened. A bucket is a file inside each partition: `HASH(customer_id) % 32` decides which of the 32 files a row lands in. Pruning happens at the directory level; bucketing pays off at the file level, mainly for joins and sampling.

Worked example: 500 million orders, 900 days, 32 buckets. Partitioning by `order_date` gives about 900 directories, so a one-day query reads roughly 1/900 of the data. Bucketing by `customer_id` adds 32 files per partition and lets a join on `customer_id` between two 32-bucket tables run as a bucket map join with no shuffle. Common mistake: bucketing a column you never join or sample on, or picking 1000 buckets so each file is smaller than one HDFS block.

```sql
-- Partitioned by date for pruning, bucketed by customer for map-side joins
CREATE TABLE orders (
  order_id     BIGINT,
  customer_id  BIGINT,
  amount       DECIMAL(12,2),
  status       STRING
)
PARTITIONED BY (order_date DATE)
CLUSTERED BY (customer_id) INTO 32 BUCKETS
STORED AS ORC
TBLPROPERTIES ('orc.compress' = 'ZLIB');
```

Partition pruning happens before any file is opened, because the partition value lives in the directory name:

In [ ]:
# Prove pruning: corrupt a file that a one-day query should never open.
victim = next((orders_dir / "order_date=2026-10-05").glob("*.parquet"))
victim.write_bytes(b"not a parquet file")
orders_glob = f"{orders_dir}/*/*.parquet"

rows, revenue = con.execute(f"""
    SELECT count(*), sum(amount)
    FROM read_parquet('{orders_glob}', hive_partitioning = true)
    WHERE order_date = DATE '2026-10-02'
""").fetchone()
assert rows == 200  # the corrupt partition was skipped from the directory name alone

try:
    con.execute(f"SELECT count(*) FROM read_parquet('{orders_glob}', hive_partitioning = true)").fetchone()
    raise AssertionError("a full scan should have opened the corrupt file")
except duckdb.Error as exc:
    print("full scan fails:", type(exc).__name__)
print(f"one-day query read 1 of 5 partitions: {rows} rows, revenue {revenue}")

### Designing partitions

Partition on the column that appears in almost every filter, usually a date, and keep the target partition size in the hundreds of MB to a few GB. Too few partitions and each query scans too much; too many and you pay metastore and filesystem overhead per partition. A useful test is how many partitions a typical query touches. If the answer is hundreds of thousands, the key is too fine.

Dynamic partitioning writes one directory per distinct value, so it lands a load without listing values in advance. It needs `hive.exec.dynamic.partition.mode=nonstrict`, because strict mode requires at least one static partition as a guard against accidentally creating millions of directories. The per-node and total limits are the safety brakes.

```sql
SET hive.exec.dynamic.partition = true;
SET hive.exec.dynamic.partition.mode = nonstrict;
SET hive.exec.max.dynamic.partitions = 5000;
SET hive.exec.max.dynamic.partitions.pernode = 500;

INSERT OVERWRITE TABLE fact_orders PARTITION (order_date)
SELECT order_id, customer_id, amount, status, order_date
FROM staging_orders;

SET hive.merge.mapfiles = true;
SET hive.merge.mapredfiles = true;
SET hive.merge.size.per.task = 268435456;
SET hive.merge.smallfiles.avgsize = 134217728;

ALTER TABLE fact_orders PARTITION (order_date = '2026-10-04') CONCATENATE;
```
Common mistake: partitioning by a high-cardinality key such as `user_id` or a raw timestamp. You get millions of tiny directories, slow planning, and object-store listing costs that dwarf the data. Bucket that key instead.

### Bucketing and bucket map joins

Bucketing writes rows into a fixed number of files per partition using a hash of the bucket columns. The hash is deterministic, so a row with the same key always lands in the same bucket index across tables with the same bucket count. That property is what makes bucket map joins possible: the join becomes a per-bucket merge with no shuffle. It also makes `TABLESAMPLE(BUCKET x OUT OF y)` cheap because the engine knows which files hold which fraction.

```sql
CREATE TABLE orders (
  order_id BIGINT, customer_id BIGINT, amount DECIMAL(12,2)
)
PARTITIONED BY (order_date DATE)
CLUSTERED BY (customer_id) INTO 64 BUCKETS
STORED AS ORC;

CREATE TABLE customers (
  customer_id BIGINT, country STRING
)
CLUSTERED BY (customer_id) INTO 64 BUCKETS
STORED AS ORC;

-- Bucket map join: both sides have matching bucket counts
SET hive.auto.convert.join = true;
SET hive.optimize.bucketmapjoin = true;
SELECT c.country, SUM(o.amount)
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.country;
```
Common mistake: bucketing after the fact. `INSERT` respects the table definition, but `LOAD DATA` and plain file drops do not, so files written into a bucketed table by hand may not be hashed correctly and the join silently falls back to a shuffle.

Buckets matter for one specific reason: joining two tables bucketed on the join key with the same bucket count lets Hive align bucket i with bucket i and skip the shuffle. With `SORTED BY` on both sides it can use a sort-merge bucket join. Choose the bucket count so each bucket file is near one HDFS block after compression; powers of two keep the modulo cheap, and 16 to 256 buckets is a common range.

Why matching bucket counts remove the shuffle: rows with the same key always hash to the same bucket index in both tables.

In [ ]:
BUCKETS = 4
con.execute("CREATE TABLE customers AS SELECT i AS customer_id, ['US', 'DE', 'IN'][1 + i % 3] AS country FROM range(50) t(i)")
# CLUSTERED BY (customer_id) INTO 4 BUCKETS: the same hash and count on both sides.
for source, target in [("staging_orders", "orders_b"), ("customers", "customers_b")]:
    con.execute(f"CREATE TABLE {target} AS SELECT *, hash(customer_id) % {BUCKETS} AS bucket FROM {source}")

full_join = con.execute("""
    SELECT c.country, sum(o.amount) FROM orders_b o JOIN customers_b c USING (customer_id)
    GROUP BY 1 ORDER BY 1
""").fetchall()
bucket_join = con.execute("""
    SELECT c.country, sum(o.amount) FROM orders_b o
    JOIN customers_b c ON o.customer_id = c.customer_id AND o.bucket = c.bucket
    GROUP BY 1 ORDER BY 1
""").fetchall()
cross_bucket_matches = con.execute("""
    SELECT count(*) FROM orders_b o
    JOIN customers_b c ON o.customer_id = c.customer_id AND o.bucket <> c.bucket
""").fetchone()[0]

assert bucket_join == full_join
assert cross_bucket_matches == 0  # bucket i only ever needs bucket i: no shuffle
print(full_join)

### Skewed tables

Skewed tables declare the values that dominate a column so Hive can store them separately. With `STORED AS DIRECTORIES` this becomes list bucketing: the hot values get their own directories and files, and each file holds rows with a single value. A join or aggregation on that key then avoids one enormous reducer.

```sql
CREATE TABLE orders (
  order_id BIGINT, customer_id BIGINT, amount DECIMAL(12,2)
)
SKEWED BY (customer_id) ON ((42), (99))
STORED AS DIRECTORIES;

SET hive.optimize.skewjoin = true;
SET hive.skewjoin.key = 100000;
```
Common mistake: declaring skew without knowing the data. Find hot keys with a grouped count and `ORDER BY ... DESC LIMIT`, then list only those values. `SKEWED BY` does not help for skew arriving after the table is created.

### File formats and compression

Start from the workload, not the format list. Analytic scans want a columnar format with per-column encoding and stored min/max: ORC or Parquet. Row-oriented work such as full-row exports or write-once raw landing is fine as text or Avro. Inside Hive, ORC is the safer default because ACID tables, LLAP, vectorized reads, and `CONCATENATE` are ORC-first. Parquet is the better choice when Spark, Trino, or another external reader is the main consumer and you want one copy of the table.

The format decides what the reader can skip. ORC stores a stripe index (min, max, sum, hasNull) plus optional bloom filters, and Parquet stores page-level statistics in the footer. Predicate pushdown uses those values to skip stripes, row groups, and pages. Compression then trades CPU for bytes: ZLIB compresses more, SNAPPY is faster, and ZSTD sits between them.

```text
format    layout                skip unit        best for
ORC       stripes + row groups  stripe, row grp  Hive-native, ACID, LLAP
Parquet   row groups + pages    row grp, page    cross-engine reads, Spark
Avro      row blocks            block            write-once rows, streaming
Text/CSV  lines                 none             raw landing only
```

```sql
CREATE TABLE fact_orders (
  order_id    BIGINT,
  customer_id BIGINT,
  amount      DECIMAL(12,2),
  status      STRING
)
PARTITIONED BY (order_date DATE)
STORED AS ORC
TBLPROPERTIES (
  'orc.compress' = 'ZLIB',
  'orc.bloom.filter.columns' = 'customer_id',
  'orc.bloom.filter.fpp' = '0.05'
);

SET hive.vectorized.execution.enabled = true;
SET hive.optimize.ppd = true;
EXPLAIN SELECT SUM(amount) FROM fact_orders WHERE status = 'paid';
```
Common mistake: adding bloom filters to every column. Bloom filters cost space and write CPU, so restrict them to high-cardinality columns that appear in filters or joins.

### SerDes

A SerDe is a pair of classes: a Serializer that turns a row object into bytes and a Deserializer that turns bytes into a row. It is wired to an `ObjectInspector` that tells Hive the field names, types, and how to read each field lazily. Because the SerDe is chosen per table, the same files can be read through different schemas.

```sql
CREATE TABLE logs (
  host  STRING,
  ts    STRING,
  level STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.RegexSerDe'
WITH SERDEPROPERTIES ('input.regex' = '^(\\S+)\\s+(\\S+)\\s+(\\S+).*$')
STORED AS TEXTFILE;
```
Common mistake: pairing a regex SerDe with a columnar format, or expecting the SerDe to fix malformed rows. A row that does not match the regex becomes all `NULL` fields rather than an error, so validate raw data before creating the table.

Common choices: `LazySimpleSerDe` for delimited text, `OpenCSVSerDe` for quoted CSV, `RegexSerDe` for fixed patterns, `JsonSerDe` for JSON, `AvroSerDe` for Avro, and `ParquetHiveSerDe` or the ORC SerDe for columnar files. Columnar formats carry their own schema, so the SerDe mostly wires the reader rather than parsing bytes field by field.

```sql
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar'     = '"',
  'escapeChar'    = '\\'
);
```
Common mistake: using `LazySimpleSerDe` for quoted CSV. A comma inside a quoted field shifts every later column. Use `OpenCSVSerde`, or better, convert to ORC once at ingest.

### Complex types and table-generating functions

Hive supports `ARRAY`, `MAP`, and `STRUCT`, plus `UNIONTYPE`. `ARRAY` is an ordered list of one type, `MAP` is key to value, and `STRUCT` is a named tuple. They come from the object model rather than the relational one, and each field is accessed with bracket or dot notation.

```sql
CREATE TABLE events (
  user_id BIGINT,
  tags    ARRAY<STRING>,
  props   MAP<STRING, STRING>,
  address STRUCT<city: STRING, zip: STRING>
)
STORED AS ORC;

SELECT user_id, tags[0], props['source'], address.city
FROM events
WHERE array_contains(tags, 'sale');
```
Common mistake: exploding a large array early in a query and multiplying rows. Filter the array first, or use `LATERAL VIEW explode` after the aggregate whenever possible.

A table generating function (UDTF) turns one input row into many output rows. Built-ins include `explode`, `posexplode`, `inline`, `stack`, `json_tuple`, and `parse_url_tuple`. They must be used in the `SELECT` list, usually with `LATERAL VIEW`, because the output changes the row count.

```sql
SELECT order_id, item
FROM orders
LATERAL VIEW explode(items) t AS item;

SELECT id, name, age
FROM users
LATERAL VIEW inline(family) f AS name, age;
```
Common mistake: mixing a UDTF with other expressions in the same `SELECT` without a `LATERAL VIEW`. Hive rejects it or produces confusing output. Wrap the UDTF in `LATERAL VIEW` and reference its alias.

In [ ]:
con.execute("""
    CREATE TABLE events AS SELECT * FROM (VALUES
        (1, ['sale', 'mobile'], MAP {'source': 'ads'}, {'city': 'Austin', 'zip': '78701'}),
        (2, ['view'],           MAP {'source': 'seo'}, {'city': 'Berlin', 'zip': '10115'})
    ) t(user_id, tags, props, address)
""")
# Hive: tags[0], props['source'], address.city  (DuckDB lists are 1-based)
picked = con.execute("""
    SELECT user_id, tags[1], props['source'], address.city
    FROM events WHERE list_contains(tags, 'sale')
""").fetchall()
# Hive: SELECT user_id, tag FROM events LATERAL VIEW explode(tags) t AS tag
exploded = con.execute("SELECT user_id, unnest(tags) AS tag FROM events ORDER BY ALL").fetchall()

assert picked == [(1, "sale", "ads", "Austin")]
assert len(exploded) == 3  # one input row per array element: UDTFs change the row count
print(picked, exploded)

## Writing data

### LOAD DATA

`LOAD DATA INPATH` moves files from HDFS into the table location; the source files disappear from their original path. `LOAD DATA LOCAL INPATH` copies from the client's local filesystem into HDFS. Neither parses or transforms the bytes, and neither runs a MapReduce job, so both are fast metadata plus file operations.

```sql
LOAD DATA INPATH '/data/staging/orders/2026-10-04/'
INTO TABLE orders PARTITION (order_date = '2026-10-04');

LOAD DATA LOCAL INPATH '/tmp/seed.csv'
INTO TABLE orders PARTITION (order_date = '2026-10-04');
```
Common mistake: expecting `LOAD DATA` to validate the file against the table schema. It does not; a mismatched delimiter or column count produces rows of `NULL` or shifted fields, discovered only at query time.

### INSERT OVERWRITE and idempotent loads

`CREATE TABLE` itself has no overwrite clause. Overwrite belongs to writes: `INSERT OVERWRITE` replaces the target data, and for a partitioned table it replaces only the partitions present in the result. The replacement is done by moving new files into place, so readers see either the old or the new partition, not a mix. `INSERT INTO` appends instead.

```sql
INSERT OVERWRITE TABLE orders PARTITION (order_date = '2026-10-04')
SELECT order_id, customer_id, amount
FROM staging_orders
WHERE order_date = '2026-10-04';

INSERT INTO TABLE orders PARTITION (order_date = '2026-10-04')
SELECT order_id, customer_id, amount FROM late_arrivals;
```
Common mistake: rerunning an `INSERT INTO` job after a failure. Each retry appends another copy of the rows. Batch loads should be idempotent, so use `INSERT OVERWRITE` for a partition or `DELETE`/`MERGE` for ACID tables.

### Changing a column type

Metadata changes are cheap because Hive is schema-on-read: `ALTER TABLE ... CHANGE COLUMN` rewrites the metastore entry only. Existing files keep their original bytes, so a widening change is safe while a narrowing or incompatible change can make old data unreadable at query time.

```sql
ALTER TABLE orders CHANGE COLUMN amount amount DECIMAL(14,2);
SET hive.metastore.disallow.incompatible.col.type.changes = true;
```
Common mistake: assuming the data was rewritten. To actually rewrite, run `INSERT OVERWRITE` into a new table with the new type. Setting the guard above makes Hive reject changes it knows are unsafe.

### ACID tables: row-level writes, not OLTP

Hive ACID tables add row-level `INSERT`, `UPDATE`, `DELETE`, and `MERGE`, but each write creates delta files that a background compactor later merges into base files. Readers must merge base and delta on the fly. That design gives seconds-to-minutes latency, not the milliseconds an OLTP system needs.

```sql
SHOW COMPACTIONS;
ALTER TABLE customers COMPACT 'MAJOR';
```
Common mistake: reading `transactional=true` as "Hive is now a database for point updates". It is still a batch warehouse that happens to accept row-level writes. Use HBase, Cassandra, or a relational database for transactional workloads.

```sql
-- ACID tables add row-level operations, but Hive still has batch latency
CREATE TABLE customers (
  id   BIGINT,
  name STRING,
  tier STRING
)
CLUSTERED BY (id) INTO 8 BUCKETS
STORED AS ORC
TBLPROPERTIES ('transactional' = 'true');

UPDATE customers SET tier = 'gold' WHERE id = 42;
DELETE FROM customers WHERE id = 43;

MERGE INTO customers AS t
USING customer_updates AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.tier = s.tier
WHEN NOT MATCHED THEN INSERT VALUES (s.id, s.name, s.tier);
```

An ACID table stores rows in base files plus delta files for each write. Readers merge them at read time, so read cost grows with the number of deltas. Compaction fixes that. Minor compaction rewrites a set of delta files into one delta; major compaction rewrites base plus deltas into a new base. Hive triggers it with `hive.compactor.delta.num.threshold` (default 10) and `hive.compactor.delta.pct.threshold` (default 0.1), and `hive.compactor.initiator.on` must be true on the metastore for automatic runs.

```sql
CREATE TABLE accounts (
  id BIGINT, balance DECIMAL(14,2), tier STRING
)
CLUSTERED BY (id) INTO 8 BUCKETS
STORED AS ORC
TBLPROPERTIES ('transactional' = 'true', 'transactional_properties' = 'insert_only');

SHOW COMPACTIONS;
ALTER TABLE accounts COMPACT 'MINOR';
ALTER TABLE accounts COMPACT 'MAJOR';
```
Common mistake: enabling ACID on a table that is only ever bulk-loaded. You pay delta files and compaction for no benefit. Use `transactional_properties = 'insert_only'` when you only append, or leave the table non-transactional.

### DROP, IF EXISTS and PURGE

`IF EXISTS` makes `DROP` idempotent: dropping a missing table or partition succeeds instead of raising an error. That matters in rerunnable ETL scripts where the first run creates state and later runs clean up.

```sql
DROP TABLE IF EXISTS staging_orders;
DROP TABLE IF EXISTS retail.staging_orders PURGE;
ALTER TABLE orders DROP IF EXISTS PARTITION (order_date = '2026-10-04');
```
Common mistake: reaching for `IF EXISTS` to hide a real bug. If a job expected the table and it was absent, the drop silently succeeds and the next step fails later with a confusing error. Check existence explicitly when the absence is unexpected.

Normally a dropped managed table or partition moves its files to the HDFS trash, so they can be restored for the retention period. `PURGE` skips the trash and deletes the bytes immediately. It applies to managed tables; external table data is never deleted by `DROP`.

```sql
DROP TABLE staging_orders PURGE;
ALTER TABLE orders DROP PARTITION (order_date = '2026-09-01') PURGE;
```
Common mistake: using `PURGE` in a pipeline that may need to rerun and recover, or expecting it to reclaim space for an external table. For externals, remove the files explicitly.

### Reading nested directories

By default a Hive table reads the files in its location, often only one level deep. To read nested directories you enable recursive input in both the mapreduce layer and the Hive layer, or you declare the subdirectories as partitions so each path is a known partition.

```sql
SET mapreduce.input.fileinputformat.input.dir.recursive = true;
SET hive.mapred.supports.subdirectories = true;

-- Preferred: make each directory an explicit partition
MSCK REPAIR TABLE orders;
SELECT COUNT(*) FROM orders WHERE order_date >= '2026-01-01';
```
Common mistake: turning on recursive reads and then querying a table whose directory also holds temporary or backup subdirectories. The query silently reads those files too. Partitions give the same reach with a clear boundary.

## Query execution and tuning

### Performance checklist

File layout first, then engine, then query shape. Fixing layout means columnar storage, right-sized partitions, and files near one HDFS block (128 or 256 MB). Engine work means Tez or Spark instead of MapReduce, vectorized execution, and the cost-based optimizer. Query shape means selecting only needed columns, filtering before joins, and letting map joins replace shuffles.

```sql
SET hive.execution.engine = tez;
SET hive.vectorized.execution.enabled = true;
SET hive.vectorized.execution.reduce.enabled = true;
SET hive.cbo.enable = true;
SET hive.auto.convert.join = true;
SET hive.merge.mapfiles = true;
SET hive.merge.size.per.task = 268435456;   -- merge targets 256 MB files

ANALYZE TABLE orders COMPUTE STATISTICS FOR COLUMNS;
ANALYZE TABLE customers COMPUTE STATISTICS FOR COLUMNS;
```
Common mistake: tuning JVM heap and container counts before checking the plan. Run `EXPLAIN` and look for full scans, a common join that should be a map join, and small-file counts. A container tweak rarely beats fixing layout.

```sql
EXPLAIN
SELECT c.country, SUM(o.amount) AS revenue
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_date = DATE '2026-10-04'
GROUP BY c.country;
-- Read the plan for: Tez/MapReduce stages, Map Join vs common join,
-- predicate pushdown and the estimated rows at each operator.
```

### Avoiding MapReduce: Tez, Spark and fetch tasks

By executing on Tez or Spark instead. Tez runs the query as one directed acyclic graph of vertices and edges, so intermediate stages avoid the mandatory write to HDFS that MapReduce requires between map and reduce. Spark offers the same benefit with its own runtime. Hive can also answer metadata-only queries, such as a partition count, from the metastore without launching a job at all.

```sql
SET hive.execution.engine = tez;     -- or spark, or mr
EXPLAIN SELECT COUNT(*) FROM orders;
-- Tez plans show a DAG of vertices; mr shows separate map and reduce stages
```
Common mistake: leaving the engine at `mr` and tuning everything else. Switching to Tez is often the single largest speedup for multi-stage HiveQL, because it removes the extra HDFS writes between stages.

```sql
SET hive.fetch.task.conversion = more;
SELECT * FROM orders WHERE order_date = DATE '2026-10-04' LIMIT 10;
-- With 'more', a simple projection/filter/limit runs as a FETCH task and
-- avoids the overhead of launching a MapReduce or Tez job.
```

### SELECT optimizations

Prune partitions in the `WHERE` clause, project only the columns you need, filter before joining, and prefer map joins over shuffles. Push predicates down so the storage layer skips stripes and pages. Avoid functions on partition columns, since `to_date(order_date) = ...` defeats pruning, and avoid `SELECT *` on wide columnar tables because it reads every column.

```sql
SET hive.cbo.enable = true;
SET hive.optimize.ppd = true;
SET hive.vectorized.execution.enabled = true;
SET hive.auto.convert.join = true;

SELECT o.order_date, SUM(o.amount)
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_date BETWEEN DATE '2026-10-01' AND DATE '2026-10-07'
  AND c.country = 'US'
GROUP BY o.order_date;
```
Common mistake: filtering after a join instead of before. Pushing the country filter into the customer side first shrinks the shuffle, and partition pruning on `orders` removes whole days before any join work happens.

### Statistics, the CBO and join strategies

The cost-based optimizer needs numbers: row count, average row size, distinct values, and null count per column and per partition. `ANALYZE TABLE ... COMPUTE STATISTICS` gathers table and partition basics; `FOR COLUMNS` adds column-level statistics. `hive.stats.autogather` collects some of this during inserts. Without statistics the optimizer guesses, and a wrong guess shows up as a bad join on a large table.

With statistics, the optimizer picks a join strategy. A map join broadcasts the small side to every mapper and avoids the shuffle, controlled by `hive.auto.convert.join` and bounded by `hive.auto.convert.join.noconditionaltask.size`. A common join sorts and merges through reducers. A bucket map join uses matching buckets. A skew join splits hot keys so one reducer is not overloaded. Hints can override the choice: `/*+ MAPJOIN(t) */`, `/*+ STREAMTABLE(t) */`, and `/*+ BROADCAST(t) */`.

```sql
ANALYZE TABLE orders COMPUTE STATISTICS FOR COLUMNS customer_id, amount;
ANALYZE TABLE customers COMPUTE STATISTICS FOR COLUMNS customer_id, country;

SET hive.cbo.enable = true;
SET hive.auto.convert.join = true;
SET hive.auto.convert.join.noconditionaltask = true;
SET hive.auto.convert.join.noconditionaltask.size = 268435456;  -- 256 MB

SET hive.optimize.skewjoin = true;
SET hive.skewjoin.key = 100000;

EXPLAIN
SELECT /*+ MAPJOIN(c) */ c.country, SUM(o.amount)
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.country;
```
Common mistake: setting a huge `noconditionaltask.size` so a genuinely large table is broadcast, then hitting out-of-memory errors. Keep the threshold near the real small-table size and refresh statistics after big loads.

```text
plan node to look for            meaning
TableScan with partitions        pruning worked
Map Join / BroadcastHashJoin     small side broadcast, no shuffle
Common Join / SortMergeJoin      full shuffle through reducers
Filter with pushed predicate     PPD reached the storage layer
Reducer count and skew warning   hot keys or too few reducers
```

### Out-of-memory errors in joins

Find out which side is blowing up. If a large table is being broadcast as a map join, lower `hive.auto.convert.join.noconditionaltask.size` so the small side really is small. If one reducer is handling a hot key, that is skew, and the fix is skew handling or salting rather than more memory. Only after that should you raise container memory.

```sql
SET hive.auto.convert.join = true;
SET hive.auto.convert.join.noconditionaltask.size = 134217728;  -- 128 MB
SET hive.optimize.skewjoin = true;
SET hive.skewjoin.key = 100000;

-- Check for a hot key before changing memory
SELECT customer_id, COUNT(*) AS c
FROM orders GROUP BY customer_id ORDER BY c DESC LIMIT 10;
```
Common mistake: raising `mapreduce.map.memory.mb` repeatedly while the real cause is one customer_id with millions of rows. Salt that key or split the job.

### ORDER BY, SORT BY, DISTRIBUTE BY and CLUSTER BY

`ORDER BY` guarantees a total order and therefore funnels every row through one reducer, which does not scale past a small result. `SORT BY` sorts within each reducer, so you get several sorted streams rather than one global order. `DISTRIBUTE BY key SORT BY key` is the pattern for global order per key, and `CLUSTER BY key` is shorthand for distributing and sorting on the same key.

```sql
-- Slow on large data: one reducer, one ordered stream
SELECT * FROM orders ORDER BY amount DESC LIMIT 100;

-- Scales: sort inside each reducer
SELECT * FROM orders SORT BY amount DESC;

-- Global order per customer, spread across reducers
SELECT * FROM orders DISTRIBUTE BY customer_id SORT BY customer_id, amount DESC;
```
Common mistake: forcing a global `ORDER BY` on a billion-row table just to look at the top rows. Add `LIMIT` so Hive can use a top-N plan, or sort per key and merge downstream.

`DISTRIBUTE BY` decides which reducer receives a row, using a hash of the listed columns. Used with `SORT BY` it gives ordered output per key without a global sort. It is the standard way to control reducer load, write files grouped by key, and add salt for skew handling.

```sql
INSERT OVERWRITE TABLE orders_by_customer PARTITION (country)
SELECT order_id, customer_id, amount, country
FROM orders
DISTRIBUTE BY customer_id
SORT BY customer_id, amount DESC;

-- Salting a hot key to spread reducer load
SELECT order_id, customer_id, amount
FROM orders
DISTRIBUTE BY customer_id, CAST(rand() * 8 AS INT);
```
Common mistake: confusing `DISTRIBUTE BY` with `PARTITION BY` from window functions, or assuming it sorts. It only routes rows; add `SORT BY` for order inside each reducer.

In [ ]:
import zlib

rows = [(f"c{i % 6}", (i * 37) % 101) for i in range(30)]  # (customer_id, amount)
REDUCERS = 3

# DISTRIBUTE BY customer_id: hash routes every row of a customer to one reducer.
reducers = {r: [] for r in range(REDUCERS)}
for customer, amount in rows:
    reducers[zlib.crc32(customer.encode()) % REDUCERS].append((customer, amount))

# SORT BY amount DESC: each reducer sorts only its own rows.
sort_by = [sorted(part, key=lambda row: -row[1]) for part in reducers.values()]
concatenated = [amount for part in sort_by for _, amount in part]
# ORDER BY amount DESC: a single reducer sees every row.
order_by = [amount for _, amount in sorted(rows, key=lambda row: -row[1])]

assert concatenated != order_by and sorted(concatenated, reverse=True) == order_by
for customer in {c for c, _ in rows}:
    assert sum(any(c == customer for c, _ in part) for part in reducers.values()) == 1
print("rows per reducer:", [len(part) for part in sort_by])

### Strict mode

Strict mode turns some dangerous queries into errors instead of full scans. The classic switch is `hive.mapred.mode=strict`. Hive 2 and later split it into separate checks under `hive.strict.checks.*`. They cover cartesian products, `ORDER BY` without `LIMIT`, implicit type comparison, and bucketing misuse.

```sql
SET hive.strict.checks.cartesian.product = true;
SET hive.strict.checks.orderby.no.limit = true;
SET hive.strict.checks.type.safety = true;
SET hive.strict.checks.bucketing = true;

-- Fails under the orderby check: no LIMIT on a global sort
SELECT * FROM orders ORDER BY amount DESC;
```
Common mistake: enabling strict checks in a shared `hive-site.xml` and breaking existing jobs that legitimately scan whole partitions. Roll it out per team or per session first, and read the error text; it names the exact setting.

### LLAP and object stores

LLAP is a long-lived daemon that keeps data and JIT-compiled operators in memory and serves queries with low latency. It changes tuning from one big job to a working set that stays hot: size `hive.llap.daemon.yarn.container.mb`, the executor count, and `hive.llap.io.memory.mb` for cache, and keep files small enough to cache. Vectorized execution is what makes the in-memory path fast, so leave `hive.vectorized.execution.enabled` on.

Object stores such as S3 and GCS are not HDFS. They have no real rename, so a write that renames a temporary directory becomes a copy plus delete. Listing is a billed call, so many small partitions and files are expensive. Prefer larger files, avoid `RENAME` and `TOUCH` in hot paths, and set a warehouse location on the object store.

```xml
<!-- hive-site.xml: point the warehouse at an object store -->
<configuration>
  <property>
    <name>hive.metastore.warehouse.dir</name>
    <value>s3a://lake/warehouse</value>
  </property>
  <property>
    <name>hive.exec.copyfile.maxsize</name>
    <value>33554432</value>
  </property>
</configuration>
```
Common mistake: running a dynamic-partition job that creates thousands of tiny files on S3, then wondering why the bill and latency grew. Merge files in the same job and target hundreds of MB per file.

## Clients and configuration

### Connecting applications

HiveServer2 speaks Thrift and exposes JDBC and ODBC drivers; it is the normal path for applications and BI tools. Beeline is the command-line JDBC client. The metastore has its own Thrift service (port 9083) for catalog-only access by engines such as Spark and Trino. Use HiveServer2 (port 10000 by default) for queries and the metastore only for catalog operations.

```bash
# Kerberos-authenticated session: no password on the command line
beeline -u "jdbc:hive2://hs2.example.com:10000/retail;principal=hive/_HOST@EXAMPLE.COM" \
  -e "SELECT country, COUNT(*) FROM orders GROUP BY country"
```

```python
# Python via PyHive against a real HiveServer2 endpoint
from pyhive import hive

conn = hive.Connection(host="hs2.example.com", port=10000, database="retail", auth="KERBEROS",
                       kerberos_service_name="hive")
cursor = conn.cursor()
cursor.execute("SELECT country, COUNT(*) FROM orders GROUP BY country")
print(cursor.fetchall())
```

Common mistake: pointing an application at the metastore port for SQL, or putting passwords in scripts instead of Kerberos or a secrets store.

### Configuration precedence and .hiverc

From lowest to highest: `hive-default.xml` inside the distribution, then `hive-site.xml` on the classpath, then command-line `--hiveconf` and `--hivevar`, then a session `SET`. Hadoop settings follow the same idea with `core-site.xml`, `hdfs-site.xml`, and `mapred-site.xml`. Later sources override earlier ones per key.

```bash
hive --hiveconf hive.execution.engine=tez -e "SET hive.execution.engine;"
```
```sql
SET hive.execution.engine=tez;
SET -v;            -- print all current settings
SET hive.execution.engine;
```
Common mistake: editing `hive-default.xml` in place. It is overwritten on upgrade and the change is invisible. Put overrides in `hive-site.xml` or a session `SET`.

`~/.hiverc` is a script the legacy Hive CLI runs at startup, so it is the place for session defaults such as headers and queue selection. It applies to interactive CLI sessions, not to HiveServer2 sessions, so beeline and JDBC clients do not read it.

```text
# ~/.hiverc
SET hive.cli.print.header=true;
SET hive.exec.dynamic.partition.mode=nonstrict;
SET mapreduce.job.queuename=etl;
USE retail;
```
Common mistake: putting team defaults only in one user's `.hiverc`. For anything shared, set it in `hive-site.xml` or pass `--hiveconf` explicitly, because other users and tools will not inherit it.

## Trade-offs: when to move off Hive

Hive is strong at large batch scans, shared metadata, and cheap storage over HDFS or object storage. It is weak at low-latency queries, high-concurrency serving, fine-grained updates, and interactive use. Modern features narrow some gaps: ACID tables with delta files and compaction, LLAP for in-memory caching and low-latency reads, and Tez or Spark for faster execution. The trade-offs are operational: compaction jobs must be scheduled, LLAP holds memory, and object stores change the cost model for renames and listings.

```sql
-- Check ACID health and compaction backlog
SHOW COMPACTIONS;

-- Check whether vectorized execution and LLAP are in play
SET hive.vectorized.execution.enabled;
SET hive.llap.execution.mode;   -- auto, all, none, or map
```
Common mistake: keeping Hive for a workload that turned interactive and paying for LLAP and compaction to imitate a serving database. Move those reads to an OLAP store or a cache, and keep Hive for the batch layer it is good at. On object stores specifically, avoid `RENAME` and directory listing in hot paths; they are metadata operations billed per call.

+ **Shuffle cost.** Every join and `GROUP BY` shuffles by key. Two big joins in one query means two shuffles; filtering and pre-aggregating before the join is usually the largest single win.
+ **Small files.** Thousands of tiny files per partition slow down the metastore, the planner and every task. Enable `hive.merge.*` on write and run `ALTER TABLE ... PARTITION ... CONCATENATE` for ORC.
+ **Partitioning versus bucketing.** Partition on low-cardinality filter columns (date, region); bucket on high-cardinality join keys. Over-partitioning on `user_id` creates millions of directories and is a classic production mistake.
+ **Data skew.** One hot key overloads a single reducer. Use `hive.optimize.skewjoin`, map-side joins for the small side, or salt the key.
+ **Tez or Spark over MapReduce.** Tez removes the forced write between stages and is typically 2-5x faster for the same HiveQL; MapReduce is now mainly of historical interest.
+ **ACID is not OLTP.** Hive ACID gives row-level `UPDATE`/`DELETE`/`MERGE`, but writes are still batched and readers may need `hive.txn.manager` configuration.
+ **When to move off Hive.** Move to Spark, Trino or a lakehouse table format (Iceberg, Delta, Hudi) when you need interactive latency, frequent upserts, schema evolution without rewrites, or query engines that read the same files without a Hive metastore. Keep Hive when the workload is scheduled batch SQL over a stable schema and the SQL base is large.

HBase is a distributed key-value store on HDFS: random reads and writes at row-key granularity, no SQL joins, no full scans by design for analytics. Hive is a SQL warehouse over files: batch scans, joins, aggregations, and a metastore catalog. They are complements, not alternatives.

## Things to remember

### Key numbers and limits

| Topic | Fact | Why it matters |
|---|---|---|
| Schema on read | Hive applies the schema at query time; the metastore holds the definitions, files hold the bytes | load is cheap; format mistakes surface at query time |
| Managed vs external | `DROP TABLE` on managed deletes data; on external removes only the metadata | use external for shared lake data |
| Partitioning | partition = directory; queries prune by partition filter | a missing partition predicate is a full-table scan |
| Bucketing | `CLUSTERED BY` into a fixed bucket count | join/sampling optimization; a rigid choice — hard to change later |
| Formats | ORC/Parquet give predicate pushdown and column pruning; TEXTFILE reads everything | format choice dwarfs most query tuning |
| `SORT BY` vs `ORDER BY` | `ORDER BY` forces a single reducer for total order; `SORT BY` orders within each reducer | `ORDER BY` over a large result is the classic hang |
| Engines | Tez/Spark replace MapReduce; strict mode blocks queries with no partition filter | MR being the default is a legacy smell |
| ACID | Hive 3 ACID tables require ORC, managed storage, and periodic compaction | upserts on the lake are possible but operationally priced |

### Semantics to memorize

* **The metastore is the catalog contract.** Engines like Spark, Trino, and Presto all read the same Hive metastore — table definitions there are the shared API of the Hadoop-era lake.
* **Partition on the filter column, bucket on the join column.** Partitions answer 'which days?'; buckets answer 'which join keys land together?'
* **Statistics drive the CBO.** Without `ANALYZE TABLE ... COMPUTE STATISTICS` the optimizer guesses join strategies and gets them wrong on skew.
* **Compactions are the tax on Hive ACID.** Updates and deletes accumulate deltas; uncompacted tables read slower over time.
* **Object storage changed the trade-off.** S3/ADLS as the store removed the rename-atomicity guarantee HDFS had; engines gained committers and table formats (Iceberg/Delta) grew to close the gap — which is why new builds usually skip plain Hive tables.

### Mistakes that keep coming back

* Querying without a partition filter — strict mode exists to forbid exactly this.
* Leaving tables as TEXTFILE 'because they arrived that way' — convert to ORC or Parquet once and every query gets cheaper.
* Thousands of tiny partitions — metastore and listing overhead dominate; wider partitions plus bucketing win.
* `ORDER BY` on a huge intermediate result — one reducer, one long hang; use `SORT BY`/`DISTRIBUTE BY` or cap with `LIMIT`.
* Bucketing on a column whose cardinality shifts — bucket counts are fixed at creation and skew worsens silently.
* Assuming DROP protects an external table — the data stays in HDFS/S3 by design, which surprises people in both directions.
* No table statistics — the CBO broadcasts or repartitions the wrong side of every join.

## Real world use cases

### 1. Hive warehouse on S3

A media company keeps 400 TB of event data in Parquet on S3 and defines Hive external tables over it, so storage and compute are billed separately and the data can also be queried by Athena and Spark.

```sql
CREATE EXTERNAL TABLE events (
  event_id STRING,
  user_id  BIGINT,
  event_ts TIMESTAMP
)
PARTITIONED BY (ds STRING)
STORED AS PARQUET
LOCATION 's3a://acme-lake/events/';

MSCK REPAIR TABLE events;   -- register any partitions added outside Hive
```

Why this design: external tables keep Hive as a metadata layer rather than the owner of the data, so another engine can read the same files.

### 2. Slowly changing dimensions with Hive ACID

A subscription business needs to correct and expire customer records in place. An ACID ORC table supports `MERGE`, which a plain text or Parquet table cannot.

```sql
MERGE INTO dim.customers AS t
USING stage.customer_updates AS s ON t.customer_id = s.customer_id
WHEN MATCHED AND s.op = 'D' THEN DELETE
WHEN MATCHED THEN UPDATE SET t.tier = s.tier, t.updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT VALUES (s.customer_id, s.name, s.tier, s.updated_at);
```

Why this design: the alternative is rewriting the whole dimension table each night, which does not scale past a few hundred GB.

### 3. Migrating a Hive warehouse to Spark and Iceberg

An organisation keeps the Hive metastore as the catalog but rewrites tables as Iceberg and runs the same HiveQL on Spark. Tables move one domain at a time, and legacy Hive reads continue through the metastore until the cutover finishes.

```sql
CREATE TABLE iceberg.dw.fact_orders
USING iceberg
PARTITIONED BY (days(order_date))
AS SELECT * FROM dw.fact_orders;

-- Verify row counts and checksums before switching producers
SELECT COUNT(*) FROM dw.fact_orders;
SELECT COUNT(*) FROM iceberg.dw.fact_orders;
```

Why this design: the migration is incremental and reversible, and no downstream SQL has to be rewritten on day one.

## Hive exercises

These exercises model the decisions a Hive tuning job turns on: partitioning, partition and file counts, format choice, bucketing, small-file cost, statistics and join choice, compaction, and Tez container sizing. They use only the Python standard library, so each cell runs as-is and asserts its own result.

### Try it 1. Choose partitioning from query filters

A query log lists the columns each query filters on. Choose the partition key so the most common filter prunes the most data.

Hint: The column that appears in every query is the partition key. Remaining filter columns inform clustering or bucketing, not partitioning.

Expected result: Partition key `order_date`, because it appears in every query.

In [ ]:
queries = [
    {"filters": ["order_date", "region"]},
    {"filters": ["order_date", "customer_id"]},
    {"filters": ["order_date", "status", "region"]},
]

counts = {}
for q in queries:
    for col in q["filters"]:
        counts[col] = counts.get(col, 0) + 1

partition_col = max(counts, key=counts.get)
assert partition_col == "order_date"
assert counts["order_date"] == len(queries)
print("partition key:", partition_col)
print("filter counts:", counts)

### Try it 2. Compute partition count and file count

A dataset spans two years of daily data and starts with 4 files per partition. You rewrite it so each partition holds exactly 1 merged file. Compute the partition count and the file count before and after.

Hint: Two years of daily partitions is 2 x 365. Files are partitions multiplied by files per partition.

Expected result: 730 partitions, 2920 files before, 730 files after.

In [ ]:
years = 2
days_per_year = 365
files_per_partition_before = 4

partitions = years * days_per_year
files_before = partitions * files_per_partition_before
files_after = partitions * 1

assert partitions == 730
assert files_before == 2920
assert files_after == 730
assert files_after < files_before
print("partitions:", partitions)
print("files before:", files_before)
print("files after:", files_after)

### Try it 3. Estimate ORC versus Parquet size from column stats

Estimate the stored size of a table from its column widths and a per-format compression factor. Column types decide the raw bytes per row; the format decides how much survives compression.

Hint: Sum the raw bytes per row, multiply by the row count, then apply each format factor. ORC and Parquet both store per-column, so wide never-read columns dominate the raw size.

Expected result: about 5.40 GiB raw, about 1.19 GiB in ORC, about 1.51 GiB in Parquet.

In [ ]:
rows = 50_000_000
column_bytes = {"order_id": 8, "customer_id": 8, "amount": 8, "status": 12, "notes": 80}
raw_per_row = sum(column_bytes.values())

raw_bytes = rows * raw_per_row
orc_bytes = raw_bytes * 0.22
parquet_bytes = raw_bytes * 0.28

gib = 1024 ** 3
assert raw_per_row == 116
assert round(raw_bytes / gib, 2) == 5.40
assert orc_bytes < parquet_bytes < raw_bytes
print("raw GiB:", round(raw_bytes / gib, 2))
print("ORC GiB:", round(orc_bytes / gib, 2))
print("Parquet GiB:", round(parquet_bytes / gib, 2))

### Try it 4. Decide bucketing from join keys

A warehouse joins three tables. Decide which column to bucket on and how many buckets to pick so a bucket map join is possible and each bucket file stays near one block.

Hint: Bucket on the join key shared by the tables, with the same bucket count on both sides. Target 128 to 256 MB per bucket file after compression.

Expected result: Bucket on `customer_id` with 64 buckets.

In [ ]:
joins = [
    {"left": "orders", "right": "customers", "key": "customer_id"},
    {"left": "orders", "right": "shipments", "key": "customer_id"},
    {"left": "orders", "right": "products", "key": "product_id"},
]
key_counts = {}
for j in joins:
    key_counts[j["key"]] = key_counts.get(j["key"], 0) + 1

bucket_key = max(key_counts, key=key_counts.get)
table_gb = 16
target_file_mb = 256
buckets = 1  # smallest power of two with a file under 256 MB
while table_gb * 1024 / buckets > target_file_mb and buckets < 1024:
    buckets *= 2

assert bucket_key == "customer_id"
assert key_counts["customer_id"] == 2
assert buckets == 64
assert bucket_key in joins[0]["key"]
print("bucket key:", bucket_key)
print("buckets needed:", buckets)

### Try it 5. Compute small-file cost per partition

A table has 730 partitions with 120 small files each. Estimate the NameNode memory cost and the per-file task overhead, then compute how many files remain after merging to one file per partition.

Hint: Hadoop uses roughly 150 bytes of NameNode heap per file or block object. Per-file task overhead is real work, so fewer files means fewer tasks.

Expected result: 87,600 files, about 12.5 MB of NameNode heap, 730 files after merging.

In [ ]:
partitions = 730
files_per_partition = 120
heap_bytes_per_file = 150

files_before = partitions * files_per_partition
heap_bytes = files_before * heap_bytes_per_file
files_after = partitions * 1

assert files_before == 87600
assert round(heap_bytes / 1024 ** 2, 1) == 12.5
assert files_after == 730
print("files before:", files_before)
print("NameNode heap MB:", round(heap_bytes / 1024 ** 2, 1))
print("files after merge:", files_after)

### Try it 6. Model statistics impact on join choice

Two tables join. The small table is 40 MB. The map-join threshold is 256 MB. With fresh statistics the optimizer sees 40 MB and picks a map join. Without statistics it assumes a default large size and picks a common join. Model both.

Hint: Fresh statistics give the real size. Missing statistics fall back to a default estimate, so the small side is no longer recognized as small.

Expected result: Fresh statistics give a map join; missing statistics give a common join.

In [ ]:
small_table_mb = 40
threshold_mb = 256
default_estimate_mb = 1_000_000

def choose_join(size_mb, threshold_mb):
    return "map join" if size_mb < threshold_mb else "common join"

with_stats = choose_join(small_table_mb, threshold_mb)
without_stats = choose_join(default_estimate_mb, threshold_mb)

assert with_stats == "map join"
assert without_stats == "common join"
assert small_table_mb < threshold_mb
print("with statistics:", with_stats)
print("without statistics:", without_stats)

### Try it 7. Plan compaction from delta-file counts

An ACID table has several partitions, each with a number of delta files. Minor compaction is triggered at 10 delta files. Mark each partition as compact or leave, and count how many need work.

Hint: Compare delta count to `hive.compactor.delta.num.threshold`, which defaults to 10. At or above the threshold, request a minor compaction.

Expected result: Partitions with 12 and 25 need compaction, so 2 partitions need work.

In [ ]:
delta_counts = {"2026-10-01": 3, "2026-10-02": 12, "2026-10-03": 25, "2026-10-04": 1}
delta_num_threshold = 10

plan = {part: ("minor compact" if n >= delta_num_threshold else "leave")
        for part, n in delta_counts.items()}
needs_work = sum(1 for v in plan.values() if v == "minor compact")

assert delta_num_threshold == 10
assert needs_work == 2
assert plan["2026-10-03"] == "minor compact"
assert plan["2026-10-04"] == "leave"
print("compaction plan:", plan)
print("partitions needing work:", needs_work)

### Try it 8. Size Tez containers

A Tez container is 4096 MB with a 0.8 heap fraction, so the Java heap is about 3276 MB. Given a shuffle working set of 9 GiB, compute how many such containers are needed to hold it in memory, and the total container memory.

Hint: Heap MB is container size times the heap fraction. Containers needed is the working set divided by heap, rounded up.

Expected result: Heap about 3276 MB, 3 containers, about 12,288 MB of container memory.

In [ ]:
import math

container_mb = 4096
heap_fraction = 0.8
heap_mb = container_mb * heap_fraction
working_set_mb = 9 * 1024

containers = math.ceil(working_set_mb / heap_mb)
total_container_mb = containers * container_mb

assert heap_mb == 3276.8
assert containers == 3
assert total_container_mb == 12288
print("heap MB per container:", heap_mb)
print("containers needed:", containers)
print("total container MB:", total_container_mb)